In [ ]:
import cptac

In [ ]:
cptac.get_cancer_info()

In [ ]:
cptac.get_source_options()

In [ ]:
en = cptac.Brca()
somatic_mutations = en.get_somatic_mutation('harmonized')
somatic_mutations.head()

In [ ]:
somatic_mutations.insert(loc=0, column='ID', value=somatic_mutations.index)

somatic_mutations.reset_index(drop=True, inplace=True)

In [ ]:
somatic_mutations['ID'].unique().shape

In [ ]:
list(somatic_mutations['ID'].unique())

In [ ]:
len(somatic_mutations)

In [ ]:
list(somatic_mutations.columns)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

def analyze_cptac_mutations(df):
    """
    Comprehensive analysis of CPTAC mutation data
    
    Parameters:
    df: pandas DataFrame with CPTAC mutation data
    
    Returns:
    Dictionary containing various analysis results
    """
    
    print("=== CPTAC Mutation Data Analysis ===\n")
    
    # Basic data overview
    print(f"Dataset shape: {df.shape}")
    print(f"Number of unique patients: {df['ID'].nunique()}")
    print(f"Number of unique genes: {df['Gene'].nunique()}")
    print(f"Total mutations: {len(df)}\n")
    
    results = {}
    
    # 1. Gene-level mutation frequency analysis
    print("1. GENE MUTATION FREQUENCY ANALYSIS")
    print("-" * 50)
    
    gene_mutation_counts = df['Gene'].value_counts()
    gene_patient_counts = df.groupby('Gene')['ID'].nunique().sort_values(ascending=False)
    
    # Combine gene statistics
    gene_stats = pd.DataFrame({
        'Total_Mutations': gene_mutation_counts,
        'Patients_Affected': gene_patient_counts,
        'Mutation_Rate': (gene_patient_counts / df['ID'].nunique() * 100).round(2)
    }).fillna(0)
    
    # Add COSMIC data if available
    if 'COSMIC_total_alterations_in_gene' in df.columns:
        cosmic_stats = df.groupby('Gene')['COSMIC_total_alterations_in_gene'].first()
        gene_stats['COSMIC_Total_Alterations'] = cosmic_stats
    
    # Add Cancer Gene Census info
    if 'CGC_Cancer_Somatic_Mut' in df.columns:
        cgc_somatic = df.groupby('Gene')['CGC_Cancer_Somatic_Mut'].first()
        gene_stats['CGC_Cancer_Gene'] = cgc_somatic.notna()
    
    print("Top 20 most frequently mutated genes:")
    print(gene_stats.head(20).to_string())
    print()
    
    results['gene_stats'] = gene_stats
    
    # 2. Patient-level mutation burden
    print("2. PATIENT MUTATION BURDEN ANALYSIS")
    print("-" * 50)
    
    patient_mutation_counts = df['ID'].value_counts()
    patient_stats = pd.DataFrame({
        'Patient_ID': patient_mutation_counts.index,
        'Total_Mutations': patient_mutation_counts.values
    })
    
    print(f"Average mutations per patient: {patient_stats['Total_Mutations'].mean():.2f}")
    print(f"Median mutations per patient: {patient_stats['Total_Mutations'].median():.2f}")
    print(f"Max mutations per patient: {patient_stats['Total_Mutations'].max()}")
    print(f"Min mutations per patient: {patient_stats['Total_Mutations'].min()}")
    print()
    
    results['patient_stats'] = patient_stats
    
    # 3. Variant type analysis
    if 'Variant_Type' in df.columns:
        print("3. VARIANT TYPE ANALYSIS")
        print("-" * 50)
        
        variant_counts = df['Variant_Type'].value_counts()
        print("Variant type distribution:")
        for variant, count in variant_counts.items():
            percentage = (count / len(df)) * 100
            print(f"{variant}: {count} ({percentage:.1f}%)")
        print()
        
        results['variant_types'] = variant_counts
    
    # 4. Clinical significance analysis
    if 'ClinVar_VCF_CLNSIG' in df.columns:
        print("4. CLINICAL SIGNIFICANCE ANALYSIS")
        print("-" * 50)
        
        # Clean up ClinVar significance data
        clinvar_data = df['ClinVar_VCF_CLNSIG'].dropna()
        if len(clinvar_data) > 0:
            clinvar_counts = clinvar_data.value_counts()
            print("ClinVar clinical significance distribution:")
            for sig, count in clinvar_counts.items():
                percentage = (count / len(clinvar_data)) * 100
                print(f"{sig}: {count} ({percentage:.1f}%)")
            print()
            
            # Focus on pathogenic variants
            pathogenic_masks = [
                clinvar_data.str.contains('Pathogenic', case=False, na=False),
                clinvar_data.str.contains('Likely_pathogenic', case=False, na=False)
            ]
            
            pathogenic_genes = df[df['ClinVar_VCF_CLNSIG'].str.contains('athogenic', case=False, na=False)]['Gene'].value_counts()
            if len(pathogenic_genes) > 0:
                print("Top genes with pathogenic variants:")
                print(pathogenic_genes.head(10).to_string())
                print()
                
                results['pathogenic_genes'] = pathogenic_genes
    
    # 5. Cancer Gene Census analysis
    if 'CGC_Cancer_Somatic_Mut' in df.columns:
        print("5. CANCER GENE CENSUS ANALYSIS")
        print("-" * 50)
        
        cgc_genes = df[df['CGC_Cancer_Somatic_Mut'].notna()]['Gene'].unique()
        print(f"Number of Cancer Gene Census genes in dataset: {len(cgc_genes)}")
        
        cgc_mutation_counts = df[df['CGC_Cancer_Somatic_Mut'].notna()]['Gene'].value_counts()
        print("Top 15 Cancer Gene Census genes by mutation frequency:")
        print(cgc_mutation_counts.head(15).to_string())
        print()
        
        results['cgc_genes'] = cgc_mutation_counts
    
    # 6. Create mutation burden score
    print("6. MUTATION BURDEN SCORING")
    print("-" * 50)
    
    # Create a comprehensive mutation burden score
    burden_factors = []
    
    # Factor 1: Mutation frequency in dataset
    gene_freq_score = (gene_stats['Total_Mutations'] / gene_stats['Total_Mutations'].max()).fillna(0)
    burden_factors.append(('Dataset_Frequency', gene_freq_score))
    
    # Factor 2: COSMIC alterations (if available)
    if 'COSMIC_total_alterations_in_gene' in df.columns:
        cosmic_scores = df.groupby('Gene')['COSMIC_total_alterations_in_gene'].first().fillna(0)
        cosmic_score_norm = (cosmic_scores / cosmic_scores.max()).fillna(0)
        burden_factors.append(('COSMIC_Score', cosmic_score_norm))
    
    # Factor 3: CGC status
    if 'CGC_Cancer_Somatic_Mut' in df.columns:
        cgc_score = df.groupby('Gene')['CGC_Cancer_Somatic_Mut'].first().notna().astype(int)
        burden_factors.append(('CGC_Status', cgc_score))
    
    # Combine scores
    burden_df = pd.DataFrame(dict(burden_factors))
    burden_df['Combined_Burden_Score'] = burden_df.mean(axis=1)
    burden_df = burden_df.sort_values('Combined_Burden_Score', ascending=False)
    
    print("Top 20 genes by combined mutation burden score:")
    print(burden_df.head(20).round(3).to_string())
    print()
    
    results['burden_scores'] = burden_df
    
    return results

def create_visualizations(df, results):
    """
    Create visualization plots for mutation analysis
    """
    
    plt.style.use('default')
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle('CPTAC Mutation Analysis Dashboard', fontsize=16, fontweight='bold')
    
    # Plot 1: Top mutated genes
    top_genes = results['gene_stats']['Total_Mutations'].head(15)
    axes[0, 0].barh(range(len(top_genes)), top_genes.values)
    axes[0, 0].set_yticks(range(len(top_genes)))
    axes[0, 0].set_yticklabels(top_genes.index)
    axes[0, 0].set_xlabel('Number of Mutations')
    axes[0, 0].set_title('Top 15 Most Frequently Mutated Genes')
    axes[0, 0].invert_yaxis()
    
    # Plot 2: Patient mutation burden distribution
    patient_counts = results['patient_stats']['Total_Mutations']
    axes[0, 1].hist(patient_counts, bins=30, alpha=0.7, color='skyblue', edgecolor='black')
    axes[0, 1].set_xlabel('Number of Mutations per Patient')
    axes[0, 1].set_ylabel('Number of Patients')
    axes[0, 1].set_title('Distribution of Mutations per Patient')
    axes[0, 1].axvline(patient_counts.mean(), color='red', linestyle='--', 
                       label=f'Mean: {patient_counts.mean():.1f}')
    axes[0, 1].legend()
    
    # Plot 3: Variant types (if available)
    if 'variant_types' in results:
        variant_data = results['variant_types'].head(8)
        axes[1, 0].pie(variant_data.values, labels=variant_data.index, autopct='%1.1f%%')
        axes[1, 0].set_title('Distribution of Variant Types')
    else:
        axes[1, 0].text(0.5, 0.5, 'Variant Type\nData Not Available', 
                        ha='center', va='center', transform=axes[1, 0].transAxes)
        axes[1, 0].set_title('Variant Types')
    
    # Plot 4: Mutation burden scores
    if 'burden_scores' in results:
        top_burden = results['burden_scores']['Combined_Burden_Score'].head(15)
        axes[1, 1].barh(range(len(top_burden)), top_burden.values)
        axes[1, 1].set_yticks(range(len(top_burden)))
        axes[1, 1].set_yticklabels(top_burden.index)
        axes[1, 1].set_xlabel('Combined Burden Score')
        axes[1, 1].set_title('Top 15 Genes by Mutation Burden Score')
        axes[1, 1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()

def export_results(results, filename_prefix='cptac_analysis'):
    """
    Export analysis results to CSV files
    """
    
    print("7. EXPORTING RESULTS")
    print("-" * 50)
    
    # Export gene statistics
    results['gene_stats'].to_csv(f'{filename_prefix}_gene_statistics.csv')
    print(f"✓ Gene statistics exported to {filename_prefix}_gene_statistics.csv")
    
    # Export patient statistics
    results['patient_stats'].to_csv(f'{filename_prefix}_patient_statistics.csv', index=False)
    print(f"✓ Patient statistics exported to {filename_prefix}_patient_statistics.csv")
    
    # Export burden scores
    if 'burden_scores' in results:
        results['burden_scores'].to_csv(f'{filename_prefix}_burden_scores.csv')
        print(f"✓ Mutation burden scores exported to {filename_prefix}_burden_scores.csv")
    
    # Export CGC genes if available
    if 'cgc_genes' in results:
        results['cgc_genes'].to_csv(f'{filename_prefix}_cgc_genes.csv')
        print(f"✓ Cancer Gene Census genes exported to {filename_prefix}_cgc_genes.csv")
    
    print("All results exported successfully!\n")

# Main execution function
def main(df):
    # Run analysis
    results = analyze_cptac_mutations(df)
    
    # Create visualizations
    create_visualizations(df, results)
    
    # Export results
    # export_results(results)
    
    return results

analysis_results = main(somatic_mutations)


print("Top 10 most mutated genes:")
print(analysis_results['gene_stats'].head(10))

print("Genes with highest mutation burden:")
print(analysis_results['burden_scores'].head(10))

In [ ]:
import pandas as pd
import numpy as np
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

def create_mutation_indicators(df, indicator_type='binary', min_gene_frequency=1, 
                             pathogenic_only=False, cgc_only=False):
    """
    Create a patient-gene matrix with mutation indicators
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Original CPTAC mutation data
    indicator_type : str
        Type of mutation indicator to create:
        - 'binary': 0/1 (mutated or not)
        - 'count': Number of mutations per gene per patient
        - 'burden_score': Weighted score based on mutation impact
        - 'categorical': Categories like 'none', 'low', 'medium', 'high'
    min_gene_frequency : int
        Minimum number of patients a gene must be mutated in to be included
    pathogenic_only : bool
        Only include pathogenic/likely pathogenic variants (requires ClinVar data)
    cgc_only : bool
        Only include Cancer Gene Census genes
        
    Returns:
    --------
    pandas.DataFrame : Patient-gene matrix
    """
    
    print("=== Creating Patient-Gene Mutation Matrix ===\n")
    
    # Filter data based on options
    filtered_df = df.copy()
    
    # Filter for pathogenic variants only
    if pathogenic_only and 'ClinVar_VCF_CLNSIG' in df.columns:
        pathogenic_mask = df['ClinVar_VCF_CLNSIG'].str.contains(
            'athogenic', case=False, na=False
        )
        filtered_df = filtered_df[pathogenic_mask]
        print(f"Filtered to {len(filtered_df)} pathogenic variants")
    
    # Filter for Cancer Gene Census genes only
    if cgc_only and 'CGC_Cancer_Somatic_Mut' in df.columns:
        cgc_mask = df['CGC_Cancer_Somatic_Mut'].notna()
        filtered_df = filtered_df[cgc_mask]
        print(f"Filtered to {len(filtered_df)} CGC gene variants")
    
    # Filter genes by minimum frequency
    gene_counts = filtered_df['Gene'].value_counts()
    frequent_genes = gene_counts[gene_counts >= min_gene_frequency].index
    filtered_df = filtered_df[filtered_df['Gene'].isin(frequent_genes)]
    
    print(f"Final dataset: {len(filtered_df)} mutations")
    print(f"Patients: {filtered_df['ID'].nunique()}")
    print(f"Genes: {filtered_df['Gene'].nunique()}")
    print(f"Genes filtered by frequency (≥{min_gene_frequency}): {len(frequent_genes)}")
    print()
    
    # Create the matrix based on indicator type
    if indicator_type == 'binary':
        matrix = create_binary_matrix(filtered_df)
    elif indicator_type == 'count':
        matrix = create_count_matrix(filtered_df)
    elif indicator_type == 'burden_score':
        matrix = create_burden_score_matrix(filtered_df)
    elif indicator_type == 'categorical':
        matrix = create_categorical_matrix(filtered_df)
    else:
        raise ValueError("indicator_type must be one of: 'binary', 'count', 'burden_score', 'categorical'")
    
    print(f"Matrix shape: {matrix.shape}")
    print(f"Matrix sparsity: {(matrix == 0).sum().sum() / (matrix.shape[0] * matrix.shape[1]) * 100:.1f}% zeros")
    
    return matrix

def create_binary_matrix(df):
    """Create binary mutation matrix (0/1)"""
    print("Creating binary mutation matrix (0=no mutation, 1=mutation)")
    
    # Create pivot table with binary indicators
    matrix = df.groupby(['ID', 'Gene']).size().unstack(fill_value=0)
    matrix = (matrix > 0).astype(int)  # Convert counts to binary
    
    return matrix

def create_count_matrix(df):
    """Create mutation count matrix"""
    print("Creating mutation count matrix (number of mutations per gene per patient)")
    
    # Count mutations per patient-gene combination
    matrix = df.groupby(['ID', 'Gene']).size().unstack(fill_value=0)
    
    return matrix

def create_burden_score_matrix(df):
    """Create burden score matrix with weighted mutations"""
    print("Creating burden score matrix (weighted by mutation impact)")
    
    # Calculate weights for different mutation types
    mutation_weights = {}
    
    # Weight by variant type if available
    if 'Variant_Type' in df.columns:
        variant_weights = {
            'Missense_Mutation': 1.0,
            'Nonsense_Mutation': 2.0,
            'Frame_Shift_Del': 2.0,
            'Frame_Shift_Ins': 2.0,
            'Splice_Site': 1.5,
            'In_Frame_Del': 1.2,
            'In_Frame_Ins': 1.2,
            'Silent': 0.1,
            'RNA': 0.5,
            'Intron': 0.2,
            '3\'UTR': 0.3,
            '5\'UTR': 0.3,
            'IGR': 0.1
        }
        df['variant_weight'] = df['Variant_Type'].map(variant_weights).fillna(1.0)
    else:
        df['variant_weight'] = 1.0
    
    # Weight by clinical significance if available
    if 'ClinVar_VCF_CLNSIG' in df.columns:
        clinvar_weights = {
            'Pathogenic': 3.0,
            'Likely_pathogenic': 2.0,
            'Uncertain_significance': 1.0,
            'Likely_benign': 0.3,
            'Benign': 0.1
        }
        
        df['clinvar_weight'] = 1.0
        for sig, weight in clinvar_weights.items():
            mask = df['ClinVar_VCF_CLNSIG'].str.contains(sig, case=False, na=False)
            df.loc[mask, 'clinvar_weight'] = weight
    else:
        df['clinvar_weight'] = 1.0
    
    # Weight by COSMIC frequency if available
    if 'COSMIC_total_alterations_in_gene' in df.columns:
        cosmic_max = df['COSMIC_total_alterations_in_gene'].max()
        df['cosmic_weight'] = 1 + (df['COSMIC_total_alterations_in_gene'].fillna(0) / cosmic_max)
    else:
        df['cosmic_weight'] = 1.0
    
    # Calculate combined weight
    df['combined_weight'] = df['variant_weight'] * df['clinvar_weight'] * df['cosmic_weight']
    
    # Create weighted matrix
    weighted_mutations = df.groupby(['ID', 'Gene'])['combined_weight'].sum().unstack(fill_value=0)
    
    return weighted_mutations

def create_categorical_matrix(df):
    """Create categorical mutation matrix"""
    print("Creating categorical mutation matrix (none/low/medium/high)")
    
    # First create count matrix
    count_matrix = df.groupby(['ID', 'Gene']).size().unstack(fill_value=0)
    
    # Define thresholds for categories
    categorical_matrix = count_matrix.copy()
    
    # Convert to categories based on mutation count
    categorical_matrix = categorical_matrix.applymap(lambda x: 
        'none' if x == 0 else
        'low' if x == 1 else
        'medium' if x <= 3 else
        'high'
    )
    
    return categorical_matrix

def add_patient_metadata(matrix, df):
    """Add patient metadata columns to the matrix"""
    
    # Get cohort information if available
    patient_info = df.groupby('ID').agg({
        'COHORT': 'first' if 'COHORT' in df.columns else lambda x: 'Unknown'
    }).fillna('Unknown')
    
    # Add total mutation count per patient
    patient_info['Total_Mutations'] = df.groupby('ID').size()
    
    # Merge with matrix
    matrix_with_metadata = matrix.join(patient_info, how='left')
    
    return matrix_with_metadata

def analyze_matrix_quality(matrix):
    """Analyze the quality and characteristics of the mutation matrix"""
    
    print("\n=== Matrix Quality Analysis ===")
    print("-" * 40)
    
    # Basic statistics
    print(f"Matrix dimensions: {matrix.shape[0]} patients × {matrix.shape[1]} genes")
    
    # Sparsity analysis
    if matrix.dtypes.iloc[0] in [int, float]:  # Numeric matrix
        zero_percentage = (matrix == 0).sum().sum() / (matrix.shape[0] * matrix.shape[1]) * 100
        print(f"Sparsity: {zero_percentage:.1f}% zeros")
        
        # Mutation statistics
        total_mutations = matrix.sum().sum()
        print(f"Total mutations: {total_mutations}")
        print(f"Average mutations per patient: {matrix.sum(axis=1).mean():.2f}")
        print(f"Average mutations per gene: {matrix.sum(axis=0).mean():.2f}")
        
        # Most frequently mutated genes
        gene_frequencies = matrix.sum(axis=0).sort_values(ascending=False)
        print(f"\nTop 10 most frequently mutated genes:")
        for gene, count in gene_frequencies.head(10).items():
            percentage = (count / matrix.shape[0]) * 100
            print(f"  {gene}: {count} patients ({percentage:.1f}%)")
        
        # Patients with highest mutation burden
        patient_burdens = matrix.sum(axis=1).sort_values(ascending=False)
        print(f"\nTop 5 patients by mutation count:")
        for patient, count in patient_burdens.head(5).items():
            print(f"  {patient}: {count} mutations")
    
    else:  # Categorical matrix
        print("Categorical matrix detected")
        value_counts = matrix.stack().value_counts()
        print("Category distribution:")
        for category, count in value_counts.items():
            percentage = (count / matrix.size) * 100
            print(f"  {category}: {count} ({percentage:.1f}%)")

def save_matrix(matrix, filename, include_metadata=True, df=None):
    """Save the mutation matrix to CSV"""
    
    if include_metadata and df is not None:
        matrix_with_metadata = add_patient_metadata(matrix, df)
        matrix_with_metadata.to_csv(filename)
        print(f"Matrix with metadata saved to: {filename}")
    else:
        matrix.to_csv(filename)
        print(f"Matrix saved to: {filename}")

def create_multiple_matrices(df, output_prefix='cptac_matrix', 
                           min_gene_frequency=2, save_all=True):
    """Create multiple types of mutation matrices"""
    
    matrices = {}
    
    # Create different types of matrices
    matrix_types = {
        'binary': 'Binary mutation matrix (0/1)',
        'count': 'Mutation count matrix',
        'burden_score': 'Weighted burden score matrix',
        'categorical': 'Categorical mutation matrix'
    }
    
    for matrix_type, description in matrix_types.items():
        print(f"\n{'='*60}")
        print(f"Creating {description}")
        print('='*60)
        
        try:
            matrix = create_mutation_indicators(
                df, 
                indicator_type=matrix_type,
                min_gene_frequency=min_gene_frequency
            )
            
            matrices[matrix_type] = matrix
            
            # Analyze matrix quality
            analyze_matrix_quality(matrix)
            
            # Save matrix
            if save_all:
                filename = f"{output_prefix}_{matrix_type}.csv"
                save_matrix(matrix, filename, include_metadata=True, df=df)
            
        except Exception as e:
            print(f"Error creating {matrix_type} matrix: {str(e)}")
            continue
    
    return matrices

# Main execution function
def main(df, output_prefix='cptac_mutation_matrix', matrix_type='binary', 
         min_gene_frequency=2, pathogenic_only=False, cgc_only=False):
    """
    Main function to create patient-gene mutation matrix
    
    Parameters:
    -----------
    df : pandas.DataFrame
        CPTAC mutation data
    output_prefix : str
        Prefix for output files
    matrix_type : str
        Type of matrix to create ('binary', 'count', 'burden_score', 'categorical', 'all')
    min_gene_frequency : int
        Minimum frequency threshold for genes
    pathogenic_only : bool
        Only include pathogenic variants
    cgc_only : bool
        Only include Cancer Gene Census genes
    """
    
    if matrix_type == 'all':
        # Create all types of matrices
        matrices = create_multiple_matrices(df, output_prefix, min_gene_frequency)
        return matrices
    else:
        # Create single matrix
        matrix = create_mutation_indicators(
            df, 
            indicator_type=matrix_type,
            min_gene_frequency=min_gene_frequency,
            pathogenic_only=pathogenic_only,
            cgc_only=cgc_only
        )
        
        # Analyze and save
        analyze_matrix_quality(matrix)
        filename = f"{output_prefix}_{matrix_type}.csv"
        save_matrix(matrix, filename, include_metadata=True, df=df)
        
        return matrix

In [ ]:

# Create binary mutation matrix (most common)
binary_matrix = main(somatic_mutations, matrix_type='binary', min_gene_frequency=5)

# Create all types of matrices
all_matrices = main(somatic_mutations, matrix_type='all', min_gene_frequency=3)

# Create matrix with only pathogenic variants
pathogenic_matrix = main(somatic_mutations, matrix_type='binary', pathogenic_only=True)

# Create matrix with only Cancer Gene Census genes
cgc_matrix = main(somatic_mutations, matrix_type='binary', cgc_only=True)

In [ ]:
cp = pd.read_csv("/home/cc/PHD/dglframework/cptac/cptac_mutation_matrix_burden_score.csv")